# 01 - Data Loading, Merging and Cleaning

**Owner: Pornraksa Suksawaeng**

This notebook is the foundation of the NextBuy project. It loads the 5 raw CSV files, merges them in the correct order to minimize memory usage, investigates data quality issues, clean the dataset, and exports it as `cleaned_data.csv` for all other notebook to use.

## 0. Imports and Configurations

In [15]:
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv
load_dotenv()

USE_S3 = os.getenv('USE_S3', 'False').lower() == 'true'
S3_BUCKET = os.getenv('S3_BUCKET', '')

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(BASE_DIR, '..', 'data')

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("Pandas version:", pd.__version__)


BASE_DIR: /Users/pon/Desktop/dev/epitech/nextbuy/notebooks
DATA_DIR: /Users/pon/Desktop/dev/epitech/nextbuy/notebooks/../data
Pandas version: 2.3.3


## 1. Load CSV Files

We load all 5 files with explicit `dtype` specifications.
The `order_products` files is the heaviest (~14M rows), so correct dtypes are critical to avoid memory crash.

In [16]:
if USE_S3:
    # NOTE: To read from S3, the AWS credentials must be configured in the environment (e.g., via AWS CLI or environment variables).
    # NOTE: Import s3fs is necessary to ensure that pandas can use the S3 filesystem, even if we don't directly use s3fs in this code.
    import s3fs
    order_products = pd.read_csv(f's3://{S3_BUCKET}/order_products.csv', 
        dtype={
            'order_id': 'Int32',
            'product_id': 'Int32',
            'add_to_cart_order': 'Int16',
            'reordered': 'Int8',
        },
    )

    products = pd.read_csv(f's3://{S3_BUCKET}/products.csv',
        dtype={
            'product_id': 'Int32',
            'aisle_id': 'Int16',
            'department_id': 'Int8',
            'product_name': 'str'
        },
    )

    aisles = pd.read_csv(f's3://{S3_BUCKET}/aisles.csv')
    departments = pd.read_csv(f's3://{S3_BUCKET}/departments.csv')

    orders = pd.read_csv(f's3://{S3_BUCKET}/orders.csv',
        dtype={
            'order_id': 'Int32',
            'user_id': 'Int32',
            'order_number': 'Int16',
            'order_dow': 'Int8',
            'order_hour_of_day': 'Int8'
        },
    )
else:
    # NOTE: When reading from local files, we can directly use the file paths. The dtypes are specified to optimize memory usage, especially for the large order_products file.
    order_products = pd.read_csv(
        os.path.join(DATA_DIR, 'order_products.csv'),
        dtype={
            'order_id': 'Int32',
            'product_id': 'Int32',
            'add_to_cart_order': 'Int16',
            'reordered': 'Int8',
        },
    )

    products = pd.read_csv(
        os.path.join(DATA_DIR, 'products.csv'),
        dtype={
            'product_id': 'Int32',
            'aisle_id': 'Int16',
            'department_id': 'Int8',
            'product_name': 'str'
        }
    )

    aisles = pd.read_csv(
        os.path.join(DATA_DIR, 'aisles.csv')
    )

    departments = pd.read_csv(
        os.path.join(DATA_DIR, 'departments.csv')
    )

    orders = pd.read_csv(
        os.path.join(DATA_DIR, 'orders.csv'),
        dtype={
            'order_id': 'Int32',
            'user_id': 'Int32',
            'order_number': 'Int16',
            'order_dow': 'Int8',
            'order_hour_of_day': 'Int8'
        }
    )

print(f'order_products: {len(order_products)} lines')
print(f'products: {len(products)} lines')
print(f'aisles: {len(aisles)} lines')
print(f'departments: {len(departments)} lines')
print(f'orders: {len(orders)} lines')

order_products: 13692886 lines
products: 49688 lines
aisles: 134 lines
departments: 21 lines
orders: 1444444 lines


## 2. Merge CSV Files 

We merge the 5 files in a specific order to minimize memory usage : `order_products` → `products` → `aisles` → `departments` → `orders`

We start from the largest file (`order_products`) and progressively enrich it with reference tables. All joins are `left` to keep every product row and reveal any unmatched keys as NaN.

In [17]:
df = (order_products
    .merge(products, on='product_id', how='left')
    .merge(aisles, on='aisle_id', how='left')
    .merge(departments, on='department_id', how='left')
    .merge(orders, on='order_id', how='left')
)

print(f'Data frame shape after merging: {df.shape[0]} lines x {df.shape[1]} columns')
df.head()

Data frame shape after merging: 13692886 lines x 14 columns


,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2,33120,1,1,Organic Egg Whites,86,16,eggs,dairy eggs,202279,3,5,9,8.0
1,2,28985,2,1,Michigan Organic Kale,83,4,fresh vegetables,produce,202279,3,5,9,8.0
2,2,9327,3,0,Garlic Powder,104,13,spices seasonings,pantry,202279,3,5,9,8.0
3,2,45918,4,1,Coconut Butter,19,13,oils vinegars,pantry,202279,3,5,9,8.0
4,2,30035,5,0,Natural Sweetener,17,13,baking ingredients,pantry,202279,3,5,9,8.0


## 3. Exploration and Sanity Check

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13692886 entries, 0 to 13692885
Data columns (total 14 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                Int32  
 1   product_id              Int32  
 2   add_to_cart_order       Int16  
 3   reordered               Int8   
 4   product_name            object 
 5   aisle_id                Int16  
 6   department_id           Int8   
 7   aisle                   object 
 8   department              object 
 9   user_id                 Int32  
 10  order_number            Int16  
 11  order_dow               Int8   
 12  order_hour_of_day       Int8   
 13  days_since_prior_order  float64
dtypes: Int16(3), Int32(3), Int8(4), float64(1), object(3)
memory usage: 835.7+ MB


In [19]:
df.describe(include='all')

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,13692886.0,13692437.0,13692440.0,13692437.0,13692437,13692437.0,13692437.0,13692437,13692437,13692529.0,13692642.0,13692683.0,13692688.0,1.281389e+07
unique,<NA>,<NA>,<NA>,<NA>,49067,<NA>,<NA>,134,21,<NA>,<NA>,<NA>,<NA>,NaN
top,<NA>,<NA>,<NA>,<NA>,Banana,<NA>,<NA>,fresh fruits,produce,<NA>,<NA>,<NA>,<NA>,NaN
freq,<NA>,<NA>,<NA>,<NA>,199654,<NA>,<NA>,1539566,4005331,<NA>,<NA>,<NA>,<NA>,NaN
mean,1885226.378435,25572.837448,8.347785,0.58959,NaN,71.207989,9.918454,NaN,NaN,102953.236033,17.146044,2.741735,13.423868,1.110298e+01
std,964451.273232,14095.388562,7.125875,0.491908,NaN,38.204561,6.280598,NaN,NaN,59459.08848,17.537008,2.08984,4.246468,8.781552e+00
min,2.0,1.0,1.0,0.0,NaN,1.0,1.0,NaN,NaN,1.0,1.0,0.0,0.0,0.000000e+00
25%,816826.0,13524.0,3.0,0.0,NaN,31.0,4.0,NaN,NaN,51447.0,5.0,1.0,10.0,5.000000e+00
50%,2278485.0,25246.0,6.0,1.0,NaN,83.0,9.0,NaN,NaN,102628.0,11.0,3.0,13.0,8.000000e+00
75%,2639194.0,37923.0,11.0,1.0,NaN,107.0,16.0,NaN,NaN,154406.0,24.0,5.0,16.0,1.500000e+01


Because we use `left` joins, any rows from `order_products` whose keys has no match in a reference table will produce NaN. We investigate these before deciding how to handle them.

A high number of unmatched key would suggest a merge issue. A low number suggests the raw data is simply incomplete for a few records.

In [20]:
# Check unmatched products_ids
unmatched_products = df[df['product_name'].isnull()]['product_id'].unique()
print(f'Number of unmatched product_ids: {len(unmatched_products)}')

# Check unmatched aisle_ids
unmatched_aisles = df[df['aisle'].isnull()]['aisle_id'].unique()
print(f'Number of unmatched aisle_ids: {len(unmatched_aisles)}')

Number of unmatched product_ids: 1
Number of unmatched aisle_ids: 1


In [21]:
# Check aisle_ids with missing aisle names
print(df[df['aisle'].isna()]['aisle_id'].value_counts())


Series([], Name: count, dtype: Int64)


**Findings:**
- 1 `product_id` has no match in `products.csv`, this product was ordered but has no reference data. Its rows will be dropped.

## 4. Cleaning 

Based on the sanity check above, we apply the following cleaning steps in order:
1. Drop rows with unmatched `product_id` or `order_id`, these rows are unusable
2. Handle `days_since_prior_order` NaN, these are meaningful (first order)
3. Check for duplicates
4. Verify `add_to_cart_order` sequence integrity

### 4.1 Drop Unusable rows

We drop rows where critical join keys have no match in the reference files.
These rows cannot be enriched with product or order information and would corrupt any analysis.

In [22]:
# Drop rows with missing product names, as they represent unmatched product_ids
df = df.dropna(subset=['product_name'])

# Unmatched product_id in products.csv 
df = df.dropna(subset=['add_to_cart_order', 'reordered'])

# Unmatched order_id in orders.csv
df = df.dropna(subset=['user_id', 'order_number', 'order_dow', 'order_hour_of_day'])

### 4.2 Fill missing aisle names

We attempt to recover missing aisle names by re-reading `aisles.csv` and looking up the unmatched IDs.
If the names are found, the join issue was a dtype mismatch. If not, the data is genuinely absent.

In [23]:
missing_aisles_ids = df[df['aisle'].isna()]['aisle_id'].unique()
aisles_lookup = pd.read_csv(os.path.join(DATA_DIR, 'aisles.csv'))
missing_aisles = aisles_lookup[aisles_lookup['aisle_id'].isin(missing_aisles_ids)]

print("Missing aisle_ids with their names:", missing_aisles)

Missing aisle_ids with their names: Empty DataFrame
Columns: [aisle_id, aisle]
Index: []


### 4.3 Handle days_since_prior_order NaN

NaN in days_since_prior_order is expected and meaningful, it only occurs on a user's very first order, where there is no previous order to measure from.

We handle in two steps:
- Create a binary flag `is_first_order` to preserve information
- Fill with `0` as a neutral placeholder to satisfy dtype and ML model requirements

The `0` is not meant to be interpreted as "same day reorder", the `is_first_order` flag carries that context.

In [24]:
# before fillna
print(df['days_since_prior_order'].isna().sum())  
df['is_first_order'] = df['days_since_prior_order'].isna().astype(int)

df['days_since_prior_order'] = df['days_since_prior_order'].fillna(0).astype('float64')
print(df['days_since_prior_order'].isna().sum()) 

878860
0


### 4.4 Remove duplicates

We check for fully duplicated rows. With correct merge, there should be none, each row represent a unique product within a unique order. If duplicate exist, it signal a merge issue.

In [25]:
number_duplicate_rows = df.duplicated().sum()
print(f'Number of duplicate rows: {number_duplicate_rows}')
if number_duplicate_rows > 0:
    df = df.drop_duplicates()
    print(f'Number of duplicate rows after dropping: {df.duplicated().sum()}')

Number of duplicate rows: 0


### 4.5 Verify add_to_cart_order sequence

We verify that every order starts its cart sequence at position 1, a minimum value above 1 would suggest a missing rows in the data.

In [26]:
min_cart = df.groupby('order_id')['add_to_cart_order'].min()
print(f'Minimum value of add_to_cart_order - min global: {min_cart.min()}')
print("Correct sequence" if min_cart.min() == 1 else "Incorrect sequence")

Minimum value of add_to_cart_order - min global: 1
Correct sequence


## 5.0 Checkup before exporting

We run a final validation to confirm the dataset is clean and consistent before exporting. 

In [27]:
print(f"Rows: {df.shape[0]:,} Columns: {df.shape[1]}")

nan_counts = df.isnull().sum()
if nan_counts.sum() == 0:
    print("No NaN remaining ")
else:
    print(nan_counts[nan_counts > 0])

n_dup = df.duplicated().sum()
print(f"No duplicates" if n_dup == 0 else f"{n_dup} duplicates remaining")

print(df.dtypes)

display(df.head(10))


Rows: 13,690,540 Columns: 15
No NaN remaining 
No duplicates
order_id                    Int32
product_id                  Int32
add_to_cart_order           Int16
reordered                    Int8
product_name               object
aisle_id                    Int16
department_id                Int8
aisle                      object
department                 object
user_id                     Int32
order_number                Int16
order_dow                    Int8
order_hour_of_day            Int8
days_since_prior_order    float64
is_first_order              int64
dtype: object


,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order
0,2,33120,1,1,Organic Egg Whites,86,16,eggs,dairy eggs,202279,3,5,9,8.0,0
1,2,28985,2,1,Michigan Organic Kale,83,4,fresh vegetables,produce,202279,3,5,9,8.0,0
2,2,9327,3,0,Garlic Powder,104,13,spices seasonings,pantry,202279,3,5,9,8.0,0
3,2,45918,4,1,Coconut Butter,19,13,oils vinegars,pantry,202279,3,5,9,8.0,0
4,2,30035,5,0,Natural Sweetener,17,13,baking ingredients,pantry,202279,3,5,9,8.0,0
5,2,17794,6,1,Carrots,83,4,fresh vegetables,produce,202279,3,5,9,8.0,0
6,2,40141,7,1,Original Unflavored Gelatine Mix,105,13,doughs gelatins bake mixes,pantry,202279,3,5,9,8.0,0
7,2,1819,8,1,All Natural No Stir Creamy Almond Butter,88,13,spreads,pantry,202279,3,5,9,8.0,0
8,2,43668,9,0,Classic Blend Cole Slaw,123,4,packaged vegetables fruits,produce,202279,3,5,9,8.0,0
9,4,46842,1,0,Plain Pre-Sliced Bagels,93,3,breakfast bakery,bakery,178520,36,1,9,7.0,0


## 6.0 Export cleaned dataset

We export the cleaned dataset as `cleaned_data.csv`.

In [28]:
if USE_S3:
    import s3fs
    df.to_csv(f's3://{S3_BUCKET}/cleaned_data.csv', index=False)
    print(f'Cleaned data saved to S3 bucket: {S3_BUCKET}/cleaned_data.csv')
else:
    output_path = os.path.join(DATA_DIR, 'cleaned_data.csv')
    df.to_csv(output_path, index=False)
    print(f'Cleaned data saved to: {output_path}')

print(f'Final data frame shape: {df.shape[0]} lines x {df.shape[1]} columns')

Cleaned data saved to: /Users/pon/Desktop/dev/epitech/nextbuy/notebooks/../data/cleaned_data.csv
Final data frame shape: 13690540 lines x 15 columns
